# Fine-Grained Emotion and Intensity Classification

This notebook adds fine-grained emotion and intensity labels to the training dataset.

## 1. Setup and Imports

In [4]:
import os
import pandas as pd
import numpy as np
from transformers import pipeline
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

## 2. Load Data

In [5]:
df = pd.read_csv('final_dataset.csv', index_col=[0])

In [6]:
print(f'Dataset loaded: {len(df)} rows')
print('\nFirst few rows:')
display(df.head())

Dataset loaded: 526295 rows

First few rows:


,Sentence,Emotion,Emotion_ID
0,i just feel really helpless and heavy hearted,fear,3
1,ive enjoyed being able to slouch about relax a...,sadness,5
2,i gave up my internship with the dmrg and am f...,fear,3
3,i dont know i feel so lost,sadness,5
4,i am a kindergarten teacher and i am thoroughl...,fear,3


In [7]:
print('Dataset info:')
print(df.info())

Dataset info:
<class 'pandas.core.frame.DataFrame'>
Index: 526295 entries, 0 to 591020
Data columns (total 3 columns):
 #   Column      Non-Null Count   Dtype 
---  ------      --------------   ----- 
 0   Sentence    526295 non-null  object
 1   Emotion     526295 non-null  object
 2   Emotion_ID  526295 non-null  int64 
dtypes: int64(1), object(2)
memory usage: 16.1+ MB
None


## 3. Initialize Zero-Shot Classifier

In [17]:
print('Loading zero-shot classification model...')
classifier = pipeline(
    'zero-shot-classification',
    model='facebook/bart-large-mnli',
    device=0,  # -1 for CPU, 0 for GPU
    batch_size=128 
)

print('Model loaded successfully!')

Loading zero-shot classification model...


Device set to use cuda:0


Model loaded successfully!


# 4. Emotion Mappings

In [18]:
fine_emotion_map = {
    'happiness': ['anticipation', 'satisfaction', 'gratitude', 'admiration'],
    'sadness': ['disappointment', 'resignation', 'frustration', 'regret'],
    'fear': ['concern', 'uncertainty', 'apprehension', 'urgency'],
    'anger': ['frustration', 'resentment', 'rejection', 'impatience'],
    'surprise': ['confusion', 'amazement', 'curiosity', 'astonishment'],
    'disgust': ['contempt', 'disappointment', 'disapproval', 'dismissiveness'],
    'neutral': ['curiosity', 'neutrality', 'acceptance', 'uncertainty']
}

intensity_labels = ['weak', 'moderate', 'intense']

## 5. Create Classification function

In [19]:
def classify_emotion_and_intensity(sentence, general_emotion):
    """
    Classify fine-grained emotion and intensity for a given sentence.
    
    Args:
        sentence (str): The text to classify
        general_emotion (str): The general emotion category
    
    Returns:
        tuple: (fine_emotion, intensity)
    """
    try:
        # Get the appropriate fine-grained emotion labels
        emotion_lower = general_emotion.lower()
        fine_labels = fine_emotion_map.get(emotion_lower, [])
        
        if not fine_labels:
            return None, None
        
        # Classify fine-grained emotion
        fine_result = classifier(
            sentence, 
            fine_labels,
            multi_label=False
        )
        fine_emotion = fine_result['labels'][0]
        
        # Classify intensity
        intensity_result = classifier(
            sentence,
            intensity_labels,
            multi_label=False
        )
        intensity = intensity_result['labels'][0].replace(' emotion', '')
        
        return fine_emotion, intensity
    
    except Exception as e:
        print(f'Error processing: {sentence[:50]}... - {str(e)}')
        return None, None

## 6. Test on Sample Data

In [20]:
print('Testing on sample sentences...')

# Test on first 5 rows
test_samples = df.head(3)

for idx, row in test_samples.iterrows():
    print(f"Sample {idx + 1}:")
    print(f"  Sentence: {row['Sentence'][:80]}...")
    print(f"  General Emotion: {row['Emotion']}")
    
    fine_emotion, intensity = classify_emotion_and_intensity(
        row['Sentence'], 
        row['Emotion']
    )
    
    print(f"  -> Fine-grained Emotion: {fine_emotion}")
    print(f"  -> Intensity: {intensity}")
    print()

Testing on sample sentences...
Sample 1:
  Sentence: i just feel really helpless and heavy hearted...
  General Emotion: fear
  -> Fine-grained Emotion: concern
  -> Intensity: intense

Sample 2:
  Sentence: ive enjoyed being able to slouch about relax and unwind and frankly needed it af...
  General Emotion: sadness
  -> Fine-grained Emotion: frustration
  -> Intensity: moderate

Sample 3:
  Sentence: i gave up my internship with the dmrg and am feeling distraught...
  General Emotion: fear
  -> Fine-grained Emotion: concern
  -> Intensity: intense



## 7. Process Full Dataset

In [21]:
print(f'\nProcessing {len(df)} sentences...')

# Setup checkpoint file
checkpoint_file = './checkpoint_progress.csv'
final_output_file = './final_dataset_extended.csv'

# Check if checkpoint exists and load progress
if os.path.exists(checkpoint_file):
    print(f'Found checkpoint file. Loading previous progress...')
    df_checkpoint = pd.read_csv(checkpoint_file)
    df = df_checkpoint.copy()
    
    # Find which emotions still need processing
    processed_emotions = df[df['Fine_Emotion'].notna()]['Emotion'].unique()
    print(f'Already processed emotions: {list(processed_emotions)}')
    
    user_input = input('Continue from checkpoint? (y/n): ').lower()
    if user_input != 'y':
        df['Fine_Emotion'] = None
        df['Intensity'] = None
else:
    # Initialize columns for first run
    df['Fine_Emotion'] = None
    df['Intensity'] = None
    print('No checkpoint found. Starting fresh...')

# Group by general emotion
emotion_groups = df.groupby('Emotion')
total_emotions = len(emotion_groups)

for emotion_idx, (general_emotion, group) in enumerate(emotion_groups):
    
    # Skip if emotion already fully processed
    if df.loc[group.index, 'Fine_Emotion'].notna().all():
        print(f'\nSkipping {general_emotion} (already processed)')
        continue
    
    print(f'\n[{emotion_idx+1}/{total_emotions}] Processing {len(group)} sentences for emotion: {general_emotion}')
    
    emotion_lower = general_emotion.lower()
    fine_labels = fine_emotion_map.get(emotion_lower, [])
    
    if not fine_labels:
        df.loc[group.index, 'Fine_Emotion'] = None
        df.loc[group.index, 'Intensity'] = None
        continue
    
    sentences = group['Sentence'].tolist()
    indices = group.index.tolist()
    
    # Batch size
    batch_size = 128
    
    group_fine_emotions = []
    group_intensities = []
    
    # Process fine-grained emotions in batches
    print('  Classifying fine-grained emotions...')
    for i in tqdm(range(0, len(sentences), batch_size), desc=f'  {general_emotion} - Fine'):
        batch = sentences[i:i+batch_size]
        
        try:
            fine_results = classifier(
                batch,
                fine_labels,
                multi_label=False
            )
            
            if isinstance(fine_results, dict):
                fine_results = [fine_results]
            
            batch_fine = [res['labels'][0] for res in fine_results]
            group_fine_emotions.extend(batch_fine)
            
        except Exception as e:
            print(f'    Error in batch {i}: {str(e)}')
            group_fine_emotions.extend([None] * len(batch))
    
    # Process intensities in batches
    print('  Classifying intensities...')
    for i in tqdm(range(0, len(sentences), batch_size), desc=f'  {general_emotion} - Intensity'):
        batch = sentences[i:i+batch_size]
        
        try:
            intensity_results = classifier(
                batch,
                intensity_labels,
                multi_label=False
            )
            
            if isinstance(intensity_results, dict):
                intensity_results = [intensity_results]
            
            batch_intensity = [res['labels'][0] for res in intensity_results]
            group_intensities.extend(batch_intensity)
            
        except Exception as e:
            print(f'    Error in batch {i}: {str(e)}')
            group_intensities.extend([None] * len(batch))
    
    # Update main dataframe with this emotion group's results
    df.loc[indices, 'Fine_Emotion'] = group_fine_emotions
    df.loc[indices, 'Intensity'] = group_intensities
    
    # Save checkpoint ONLY after completing each emotion
    df.to_csv(checkpoint_file, index=False)
    print(f'  Checkpoint saved')

# Save final output
df.to_csv(final_output_file, index=False)
print(f'\nProcessing complete! Final output saved to: {final_output_file}')

# Clean up checkpoint file
if os.path.exists(checkpoint_file):
    os.remove(checkpoint_file)
    print('Checkpoint file cleaned up')


Processing 526295 sentences...
No checkpoint found. Starting fresh...

[1/7] Processing 64930 sentences for emotion: anger
  Classifying fine-grained emotions...


  anger - Fine: 100%|██████████| 508/508 [09:52<00:00,  1.17s/it]


  Classifying intensities...


  anger - Intensity: 100%|██████████| 508/508 [07:37<00:00,  1.11it/s]


  Checkpoint saved

[2/7] Processing 7571 sentences for emotion: disgust
  Classifying fine-grained emotions...


...

  Checkpoint saved

[7/7] Processing 15708 sentences for emotion: surprise
  Classifying fine-grained emotions...


  surprise - Fine: 100%|██████████| 123/123 [02:19<00:00,  1.13s/it]


  Classifying intensities...


  surprise - Intensity: 100%|██████████| 123/123 [01:45<00:00,  1.16it/s]


  Checkpoint saved

Processing complete! Final output saved to: ./final_dataset_extended.csv
Checkpoint file cleaned up


## 8. Review Results

In [22]:
print('Updated dataset preview:')
display(df.head(10))

print('\nFine-grained emotion distribution:')
print(df['Fine_Emotion'].value_counts())

print('\nIntensity distribution:')
print(df['Intensity'].value_counts())

print('\nMissing values:')
print(df[['Fine_Emotion', 'Intensity']].isnull().sum())

Updated dataset preview:


,Sentence,Emotion,Emotion_ID,Fine_Emotion,Intensity
0,i just feel really helpless and heavy hearted,fear,3,concern,intense
1,ive enjoyed being able to slouch about relax a...,sadness,5,frustration,moderate
2,i gave up my internship with the dmrg and am f...,fear,3,concern,intense
3,i dont know i feel so lost,sadness,5,frustration,intense
4,i am a kindergarten teacher and i am thoroughl...,fear,3,concern,intense
5,i was beginning to feel quite disheartened,sadness,5,disappointment,weak
6,i fear that they won t ever feel that deliciou...,happiness,4,anticipation,intense
7,im forever taking some time out to have a lie ...,surprise,6,confusion,intense
8,i can still lose the weight without feeling de...,sadness,5,resignation,moderate
9,i try to be nice though so if you get a bitchy...,happiness,4,satisfaction,intense



Fine-grained emotion distribution:
Fine_Emotion
satisfaction      88548
frustration       64029
disappointment    61582
uncertainty       53162
admiration        41193
anticipation      32689
regret            28738
resentment        23296
acceptance        22617
apprehension      19584
curiosity         19010
concern           18264
gratitude         10511
resignation        9747
impatience         5640
astonishment       4861
amazement          4732
confusion          4441
rejection          3745
disapproval        3711
neutrality         1923
urgency            1669
dismissiveness     1423
contempt           1180
Name: count, dtype: int64

Intensity distribution:
Intensity
intense     322587
moderate    157148
weak         46560
Name: count, dtype: int64

Missing values:
Fine_Emotion    0
Intensity       0
dtype: int64


## 9. Quality Checks

In [24]:
if 'Fine_Emotion' in df.columns:
    print('Checking for potential issues...')
    
    # Check if any emotions are None
    null_count = df[['Fine_Emotion', 'Intensity']].isnull().sum()
    if null_count.any():
        print(f'Warning: Found {null_count.sum()} null values')
        print(null_count)
    else:
        print('No null values found')
    
    # Show distribution by general emotion
    print('\nFine-grained emotions by general emotion category:')
    for emotion in df['Emotion'].unique():
        subset = df[df['Emotion'] == emotion]['Fine_Emotion'].value_counts()
        print(f'\n{emotion.capitalize()}:')
        print(subset)

Checking for potential issues...
No null values found

Fine-grained emotions by general emotion category:

Fear:
Fine_Emotion
apprehension    19584
concern         18264
uncertainty     10057
urgency          1669
Name: count, dtype: int64

Sadness:
Fine_Emotion
disappointment    60325
frustration       31780
regret            28738
resignation        9747
Name: count, dtype: int64

Happiness:
Fine_Emotion
satisfaction    88548
admiration      41193
anticipation    32689
gratitude       10511
Name: count, dtype: int64

Surprise:
Fine_Emotion
astonishment    4861
amazement       4732
confusion       4441
curiosity       1674
Name: count, dtype: int64

Anger:
Fine_Emotion
frustration    32249
resentment     23296
impatience      5640
rejection       3745
Name: count, dtype: int64

Neutral:
Fine_Emotion
uncertainty    43105
acceptance     22617
curiosity      17336
neutrality      1923
Name: count, dtype: int64

Disgust:
Fine_Emotion
disapproval       3711
dismissiveness    1423
disappoin